## Create a Timestamped S3 Trigger File

This code creates a new copy of the cleaned diabetes dataset in a designated Amazon S3 trigger folder. The copied file is given a unique timestamped filename so that it can be detected as a **new input event** by a downstream MLOps process, such as an S3 polling mechanism or automated SageMaker pipeline trigger.

### How the Code Works

1. **Import required libraries**

   * `boto3` is the AWS SDK for Python and is used to communicate with Amazon S3.
   * `datetime` is used to generate a timestamp for the new trigger file.

2. **Define the AWS region and S3 bucket**

   * The code connects to the `ap-southeast-1` AWS region.
   * The project data is stored in the `nyp-26s1-iti113` S3 bucket.

3. **Specify the source dataset**

   The existing cleaned diabetes dataset is located at:

   ```text
   s3://nyp-26s1-iti113/iti113/team02/data/diabetes/cleaned/diabetes_binary_cleaned_dataset.csv
   ```

   This dataset acts as the source file that will be copied.

4. **Generate a unique timestamp**

   ```python
   timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
   ```

   The timestamp uses the format:

   ```text
   YYYYMMDD_HHMMSS
   ```

   For example:

   ```text
   20260822_211500
   ```

   Using a timestamp prevents the trigger file from overwriting a previous file and allows each trigger event to be uniquely identified.

5. **Create the destination S3 key**

   The copied dataset is stored under:

   ```text
   iti113/team02/trigger/input/
   ```

   with a filename such as:

   ```text
   diabetes_binary_cleaned_dataset_trigger_20260822_211500.csv
   ```

6. **Create an S3 client**

   ```python
   s3 = boto3.client("s3", region_name=REGION)
   ```

   This initializes the connection to Amazon S3 using the AWS credentials configured in the current Jupyter environment.

7. **Copy the dataset to the trigger folder**

   ```python
   s3.copy_object(...)
   ```

   `copy_object()` performs a server-side copy within S3. The original cleaned dataset remains unchanged, while a new timestamped copy is created in the trigger input location.

8. **Display the generated trigger file**

   After the copy succeeds, the notebook prints the full S3 URI of the newly created file.

   Example:

   ```text
   Trigger file created:
   s3://nyp-26s1-iti113/iti113/team02/trigger/input/diabetes_binary_cleaned_dataset_trigger_20260822_211500.csv
   ```

### Role in the MLOps Pipeline

This step simulates the arrival of a **new dataset file** in the pipeline's monitored S3 input location.

The expected flow is:

```text
Cleaned Dataset
      ↓
Create Timestamped Copy
      ↓
S3 Trigger Input Folder
      ↓
Polling / Trigger Detection
      ↓
SageMaker Pipeline Execution
      ↓
Preprocessing → Training → Evaluation → Model Registration
```

Therefore, the code provides a simple way to test an **S3-based data-triggered MLOps workflow** without modifying or deleting the original cleaned dataset.

> **Note:** The Jupyter environment must have valid AWS credentials and sufficient IAM permissions to read from the source S3 object and write to the destination S3 path.


In [1]:
import boto3
from datetime import datetime

REGION = "ap-southeast-1"
BUCKET = "nyp-26s1-iti113"

SOURCE_KEY = (
    "iti113/team02/data/diabetes/cleaned/"
    "diabetes_binary_cleaned_dataset.csv"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

DESTINATION_KEY = (
    "iti113/team02/trigger/input/"
    f"diabetes_binary_cleaned_dataset_trigger_{timestamp}.csv"
)

s3 = boto3.client("s3", region_name=REGION)

s3.copy_object(
    Bucket=BUCKET,
    CopySource={
        "Bucket": BUCKET,
        "Key": SOURCE_KEY,
    },
    Key=DESTINATION_KEY,
)

print("Trigger file created:")
print(f"s3://{BUCKET}/{DESTINATION_KEY}")

Trigger file created:
s3://nyp-26s1-iti113/iti113/team02/trigger/input/diabetes_binary_cleaned_dataset_trigger_20260822_131302.csv
